# From Charts to Maps

**Series:** Data Visualization in Python (notebook 3 of 3)
**Time:** about 40 minutes

**In this notebook you will:**
- Turn a table into a choropleth map with GeoPandas
- See why counts must be normalized into rates before mapping
- Compare classification methods on the same data
- Use categorical colors and proportional symbols
- Check a palette for color vision deficiency
- Make an interactive map
- Optional: map Indiana Census data with your own Census API key

**You should already know:** notebooks 1a and 1b, and the Map Design episode of this module.

> **Before you start:** click **File > Save a copy in Drive** so your changes are kept.
> Then run each cell in order by clicking it and pressing **Shift + Enter**.

## The data

We use results from the **2013 Montreal mayoral election**, built into Plotly: vote counts for the three
leading candidates (Denis Coderre, Richard Bergeron and Mélanie Joly) in each of 58 electoral districts,
plus the district boundaries. Nothing needs to be downloaded.

At the end there is an **optional** section that uses U.S. Census data for Indiana. It needs a free
Census API key; skip it if you do not have one.

## Step 1: Install and import

In [ ]:
!pip install -q mapclassify

import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import plotly.express as px

## Step 2: Join the votes to the district shapes

The votes are in a table and the shapes are in GeoJSON. Both have a `district` column, so we can join
them, the same way you would join Census tables to tract boundaries.

In [ ]:
votes = px.data.election()
shapes = gpd.GeoDataFrame.from_features(px.data.election_geojson(), crs="EPSG:4326")

districts = shapes.merge(votes, on="district")
districts = districts.to_crs(epsg=32618)   # UTM zone 18N, in meters, suitable for Montreal
districts[["district", "Coderre", "Bergeron", "Joly", "total", "winner"]].head()

## Step 3: Your first choropleth map

`plot(column=...)` shades each district by a column's value.

In [ ]:
ax = districts.plot(column="total", cmap="Blues", legend=True, edgecolor="white",
                    linewidth=0.5, figsize=(7, 6),
                    legend_kwds={"label": "Total votes cast"})
ax.set_title("Total votes by district, Montreal 2013")
ax.set_axis_off()
plt.show()

## Step 4: Counts versus rates

Total votes is a **count**. Districts with more voters get more votes, so a map of Coderre's votes
mostly repeats the map of where voters are. Dividing by the total gives his **share** of the vote, which
answers the real question: where was he popular?

In [ ]:
districts["coderre_pct"] = 100 * districts["Coderre"] / districts["total"]

fig, (left, right) = plt.subplots(1, 2, figsize=(13, 6))
districts.plot(column="Coderre", cmap="Purples", legend=True, ax=left,
               edgecolor="white", linewidth=0.4, legend_kwds={"label": "Votes for Coderre"})
left.set_title("Count: votes for Coderre")
districts.plot(column="coderre_pct", cmap="Purples", legend=True, ax=right,
               edgecolor="white", linewidth=0.4, legend_kwds={"label": "Coderre's share (%)"})
right.set_title("Rate: Coderre's share of the vote")
for ax in (left, right):
    ax.set_axis_off()
plt.show()

The two maps disagree in places. The rate map is the right one for "where was he popular?". The count map
answers "where did his votes come from?", which is better shown with proportional symbols (Step 7).

## Step 5: Classification changes the story

`scheme=` groups values into classes, as in the Map Design episode. Here are three methods on the same
data. First, look at the distribution.

In [ ]:
districts["coderre_pct"].plot(kind="hist", bins=20, color="#9ecae1", edgecolor="white")
plt.xlabel("Coderre's share of the vote (%)")
plt.ylabel("Number of districts")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6))
for ax, scheme in zip(axes, ["EqualInterval", "Quantiles", "NaturalBreaks"]):
    districts.plot(column="coderre_pct", scheme=scheme, k=5, cmap="Purples",
                   legend=True, ax=ax, edgecolor="white", linewidth=0.4,
                   legend_kwds={"loc": "upper left", "bbox_to_anchor": (0.95, 1), "fontsize": 8,
                                "title": "Share (%)", "fmt": "{:.0f}"})
    ax.set_title(scheme)
    ax.set_axis_off()
plt.show()

Compare the three maps. Which districts change class? Which method would you choose, and why? There is
no single right answer, but your choice should match your message, and you should use the same breaks
when comparing several maps.

## Step 6: Categories need categorical colors

The `winner` column is a category with no order, so it needs distinct hues, not light-to-dark shades.

In [ ]:
ax = districts.plot(column="winner", categorical=True, cmap="Set2", legend=True,
                    edgecolor="white", linewidth=0.6, figsize=(7, 6),
                    legend_kwds={"title": "District winner", "loc": "upper left", "bbox_to_anchor": (1, 1)})
ax.set_title("Winning candidate by district, Montreal 2013")
ax.set_axis_off()
plt.show()

## Step 7: Proportional symbols for counts

To show **counts**, draw a circle in each district sized by the value, with circle **area**
proportional to the count.

In [ ]:
centers = districts.copy()
centers["geometry"] = districts.representative_point()

ax = districts.plot(color="#f0f0f0", edgecolor="#bbbbbb", linewidth=0.5, figsize=(7, 6))
centers.plot(ax=ax, markersize=centers["Coderre"] / 15, color="#6a51a3", alpha=0.6,
             edgecolor="white")
for votes_example in [1000, 3000, 6000]:
    ax.scatter([], [], s=votes_example / 15, color="#6a51a3", alpha=0.6, label=f"{votes_example:,}")
ax.legend(title="Votes for Coderre", loc="upper left", bbox_to_anchor=(1, 1), labelspacing=1.5,
          borderpad=1, frameon=False)
ax.set_title("Where Coderre's votes came from")
ax.set_axis_off()
plt.show()

## Step 8: Check your palette for color vision deficiency

About 1 in 12 men of Northern European ancestry has a red-green color vision deficiency. The
`colorspacious` package can simulate how a figure looks to them. Below, the same map uses a red-green
palette and viridis, then both are simulated for deuteranopia.

In [ ]:
!pip install -q colorspacious
import numpy as np
from colorspacious import cspace_convert

def render(cmap):
    fig, ax = plt.subplots(figsize=(5, 4.5))
    districts.plot(column="coderre_pct", scheme="Quantiles", k=5, cmap=cmap, ax=ax,
                   edgecolor="white", linewidth=0.4)
    ax.set_axis_off()
    fig.canvas.draw()
    image = np.asarray(fig.canvas.buffer_rgba())[..., :3] / 255
    plt.close(fig)
    return image

def deuteranopia(image):
    simulated = cspace_convert(image, {"name": "sRGB1+CVD", "cvd_type": "deuteranomaly",
                                       "severity": 100}, "sRGB1")
    return np.clip(simulated, 0, 1)

fig, axes = plt.subplots(2, 2, figsize=(10, 9))
for row, cmap in enumerate(["RdYlGn", "viridis"]):
    image = render(cmap)
    axes[row, 0].imshow(image)
    axes[row, 0].set_title(f"{cmap}: typical vision")
    axes[row, 1].imshow(deuteranopia(image))
    axes[row, 1].set_title(f"{cmap}: simulated deuteranopia")
for ax in axes.flat:
    ax.set_axis_off()
plt.show()

With `RdYlGn`, the red and green ends become similar olive tones. With `viridis`, the order from light to
dark survives. Prefer palettes like `viridis`, `cividis`, `Blues` or `Purples` for ordered data.

## Step 9: An interactive map

`explore()` turns a GeoDataFrame into a web map you can zoom and hover over. We choose an Esri basemap
because the default OpenStreetMap and CARTO backgrounds do not load inside Colab.

In [ ]:
districts.explore(column="coderre_pct", cmap="Purples", scheme="NaturalBreaks", k=5,
                  tooltip=["district", "winner", "coderre_pct"],
                  tiles="Esri.WorldGrayCanvas",
                  legend_kwds={"caption": "Coderre's share of the vote (%)"})

## Step 10 (optional): Indiana Census data with your own API key

If you have a free **Census API key** (request one at <https://api.census.gov/data/key_signup.html>),
paste it below to map **median household income** for Indiana's 92 counties from the 2023 American
Community Survey. The data and county boundaries are read straight from the Census website into memory.

If you leave the placeholder, this step is skipped and the rest of the notebook still works.

In [ ]:
import requests

CENSUS_KEY = "PASTE_YOUR_CENSUS_API_KEY_HERE"   # optional

indiana = None
if CENSUS_KEY.startswith("PASTE"):
    print("No Census API key entered, so this optional step is skipped.")
else:
    reply = requests.get(
        "https://api.census.gov/data/2023/acs/acs5",
        params={"get": "NAME,B01003_001E,B19013_001E", "for": "county:*", "in": "state:18",
                "key": CENSUS_KEY},
        timeout=60,
    )
    reply.raise_for_status()
    rows = reply.json()
    table = pd.DataFrame(rows[1:], columns=rows[0])
    table["GEOID"] = table["state"] + table["county"]
    table["population"] = pd.to_numeric(table["B01003_001E"])
    table["median_income"] = pd.to_numeric(table["B19013_001E"])

    shapes = gpd.read_file(
        "https://www2.census.gov/geo/tiger/GENZ2023/shp/cb_2023_us_county_500k.zip")
    shapes = shapes[shapes["STATEFP"] == "18"]
    indiana = shapes.merge(table[["GEOID", "NAME", "population", "median_income"]], on="GEOID")
    indiana = indiana.to_crs(epsg=26916)   # UTM zone 16N
    print(len(indiana), "Indiana counties loaded")

In [ ]:
if indiana is not None:
    ax = indiana.plot(column="median_income", scheme="NaturalBreaks", k=5, cmap="Greens",
                      legend=True, edgecolor="white", linewidth=0.5, figsize=(6, 8),
                      legend_kwds={"title": "Median household income ($)", "loc": "upper left",
                                   "bbox_to_anchor": (1, 1), "fmt": "{:,.0f}"})
    ax.set_title("Median household income by county, Indiana\n2019 to 2023 ACS estimates")
    ax.set_axis_off()
    plt.show()

Median household income is already a rate-like summary, so it can go straight on a choropleth map.
ACS values are **estimates** with margins of error; small counties have larger margins, which matters if
your map will inform decisions.

## Try it yourself

Make a map of **Mélanie Joly's share of the vote** (`Joly` divided by `total`). Use a colorblind-safe sequential palette and the classification method you think fits best, with a title and legend.

*Hint:* copy Steps 4 and 5, and change the column.

<details>
<summary><b>Show a solution</b></summary>

```python
districts["joly_pct"] = 100 * districts["Joly"] / districts["total"]
ax = districts.plot(column="joly_pct", scheme="NaturalBreaks", k=5, cmap="viridis_r",
                    legend=True, edgecolor="white", linewidth=0.4, figsize=(7, 6),
                    legend_kwds={"title": "Joly's share (%)", "loc": "upper left",
                                 "bbox_to_anchor": (1, 1), "fmt": "{:.0f}"})
ax.set_title("Mélanie Joly's share of the vote by district, Montreal 2013")
ax.set_axis_off()
plt.show()
```

</details>

In [ ]:
# Write your answer here

## Recap

- A choropleth map is a table joined to shapes, shaded by one column.
- Map rates (shares, per capita) on choropleths; show counts with proportional symbols.
- Classification methods can change the pattern; look at a histogram and choose deliberately.
- Use categorical palettes for categories and colorblind-safe sequential palettes for numbers.
- `explore()` makes an interactive map; choose a basemap that loads in your environment.